# DeepWeeds Day 2 — notebook thực nghiệm

Import notebook này vào Kaggle bằng URL GitHub, bật **GPU** và **Internet** trong Settings, rồi chạy các ô theo thứ tự. Ô đầu tự clone source. Gắn Kaggle Input `imsparsh/deepweeds` để dùng ảnh có sẵn; nếu không gắn, `prepare` tải ảnh từ Zenodo. Chỉ chọn cấu hình bằng **val**; test chỉ dùng ở ô chung kết.


In [ ]:
from pathlib import Path
import json, sys, platform, subprocess
REPO_DIR = Path.cwd()
if not (REPO_DIR / "eval.py").exists():
    REPO_DIR = Path("/kaggle/working/K4-Track4-Day2-Deeplearning-Advance") if Path("/kaggle/working").exists() else Path("/content/K4-Track4-Day2-Deeplearning-Advance")
if not (REPO_DIR / "eval.py").exists():
    subprocess.run(["git", "clone", "https://github.com/Nituv05/K4-Track4-Day2-VuThuongTin-2A202602955-Deeplearning-Advance.git", str(REPO_DIR)], check=True)
elif str(REPO_DIR).startswith(("/kaggle/working/", "/content/")):
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
assert (REPO_DIR / "eval.py").exists(), f"Clone failed or wrong path: {REPO_DIR}"
%cd {REPO_DIR}
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "submissions/deepweeds_lab/code/requirements.txt"], check=True)
import torch, timm, pandas as pd
print(platform.python_version(), torch.__version__, timm.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No CUDA GPU")


## 0. Tải và kiểm dữ liệu

Nếu đã gắn Kaggle Input `imsparsh/deepweeds`, `prepare` dùng ảnh ở đó. Nếu không, ảnh tải từ [Zenodo](https://zenodo.org/records/7939060) và kiểm MD5 `b7b30f96d466fba86016aa5a26606e0f`. `eda` in số ảnh từng tập/lớp, kiểm giao rỗng và 17.509 ảnh, rồi lưu biểu đồ phân bố và 3 ảnh mẫu mỗi lớp.

In [ ]:
def stage(name, *options):
    subprocess.run([sys.executable, "submissions/deepweeds_lab/code/experiments.py", name, *options], check=True)
stage("prepare")
stage("eda")
print((Path("submissions/deepweeds_lab") / "eda.json").read_text())


## 0b. Kiểm pipeline

Overfit một batch 8 ảnh bằng head đóng băng, kiểm Focal γ=0 ≈ CE và xem ảnh CutMix trong `curves/`.

In [ ]:
stage("sanity")


## 1. Năm backbone

B01–B05 dùng cùng fold, seed và công thức nền. Đọc `summary.json`/`results.xlsx` để so sánh macro-F1 val, tham số, GMAC và thời gian.

In [ ]:
stage("backbones")


## 2. Công thức huấn luyện

T01–T12 thay từng yếu tố của B01: khởi tạo, augmentation, loss, sampler, Mixup/CutMix và EMA. B01 có cùng cấu hình với T00.

In [ ]:
stage("training")
base=Path("submissions/deepweeds_lab/runs")
candidates=[]
for exp in ["B01","B02","B03","B05"]+[f"T{i:02d}" for i in range(1,13)]:
    p=base/exp/"seed0"/"summary.json"
    if p.exists():
        s=json.loads(p.read_text()); candidates.append((exp,s["val_macro_f1"],s["val_top1"]))
print(sorted(candidates,key=lambda row:row[1],reverse=True))
SOURCE_EXP=max(candidates,key=lambda row:row[1])[0]
print("Val-selected source:",SOURCE_EXP)

## 3. Phương pháp suy luận trên val

I00–I05 đo macro-F1, ECE và độ trễ p50/p95/p99 với warmup 10 và 100 lần đo. Chọn phương pháp dựa trên val và ngân sách độ trễ của bạn.

In [ ]:
stage("inference", "--source", SOURCE_EXP)
inf=json.loads(Path("submissions/deepweeds_lab/inference_results.json").read_text())
for r in inf: print(r["method"],r["val_macro_f1"],r["val_ece"],r["latency"]["p95"])
# Ưu tiên macro-F1, sau đó ECE và p95 khi hòa. Thay đổi quy tắc trước khi xem test nếu cần ràng buộc robot.
METHOD=sorted(inf,key=lambda r:(-r["val_macro_f1"],r["val_ece"],r["latency"]["p95"]))[0]["method"]
print("Val-selected method:",METHOD)

## 4. Chung kết: 3 seed, test một lượt mỗi seed

Chỉ chạy sau khi `SOURCE_EXP` và `METHOD` đã chốt bằng val. F01 và T00 cùng seed 0, 1, 2. `F01uncal` dùng cùng logit nếu chọn I04.

In [ ]:
stage("final", "--source", SOURCE_EXP, "--method", METHOD)
stage("export")
print("Submission files:", list(Path("submissions/deepweeds_lab").glob("*.xlsx")), list(Path("submissions/deepweeds_lab").glob("*.md")))


## 5. Chấm bằng `eval.py` gốc

Kết quả `score` phải trùng với `results.xlsx`. Dùng `grade` cho phần I rubric.

In [ ]:
PRED = "submissions/deepweeds_lab/predictions/"
common = ["--test-csv", "data/labels/test_subset0.csv", "--labels", "data/labels/labels.csv"]
for exp in ("F01", "T00"):
    subprocess.run([sys.executable, "eval.py", "score", "--pred", PRED + f"{exp}_seed*_test.csv", *common, "--tag", exp, "--out", "eval_out"], check=True)
selected = next(r for r in inf if r["method"] == METHOD)
grade = [sys.executable, "eval.py", "grade", "--final", PRED + "F01_seed*_test.csv", "--baseline", PRED + "T00_seed*_test.csv", "--final-val", PRED + "F01_seed*_val.csv", "--val-csv", "data/labels/val_subset0.csv", "--latency-p95-ms", str(selected["latency"]["p95"]), "--latency-method", "proper", *common, "--out", "eval_out"]
if METHOD == "I04":
    grade += ["--uncal", PRED + "F01uncal_seed*_test.csv"]
subprocess.run(grade, check=True)
subprocess.run([sys.executable, "submissions/deepweeds_lab/code/audit.py"], check=True)


## 6. Đóng gói bài nộp

Tạo ZIP gồm code, README, `results.xlsx`, `report.md`, đường cong và dự đoán. Tải ZIP từ bảng Files của Colab/Kaggle.


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
import shutil
submission = Path("submissions/deepweeds_lab")
if Path("eval_out").is_dir():
    shutil.copytree("eval_out", submission / "evaluation", dirs_exist_ok=True)
required = [submission/"results.xlsx", submission/"report.md"]
assert all(p.is_file() for p in required), required
archive = Path("/kaggle/working/deepweeds_submission.zip") if Path("/kaggle/working").exists() else Path("/content/deepweeds_submission.zip")
with ZipFile(archive, "w", ZIP_DEFLATED) as z:
    for p in submission.rglob("*"):
        if p.is_file() and (p.parent == submission or p.relative_to(submission).parts[0] in ("code", "curves", "predictions", "logs", "evaluation")) and p.suffix not in (".pt", ".pyc"):
            z.write(p, p.relative_to(Path.cwd()))
    z.write("eval.py", "eval.py")
print("Download:", archive, "size MB:", round(archive.stat().st_size/1e6, 1))
